# NB04b Peptide Identification

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/CSBiology/BIO-BTE-06-L-7/gh-pages?filepath=NB04b_Peptide_Identification.ipynb)

[Download Notebook](https://github.com/CSBiology/BIO-BTE-06-L-7/releases/download/NB04a_NB04b/NB04b_Peptide_Identification.ipynb)

1. Understanding peptide identification from MS2 spectra
2. Matching and scoring of Tandem MS peptide identification
    3. Step 1: Data acquisition and preprocessing
    4. Step 2: Preselecting the peptides of interest
    5. Step 3+4: Matching and Scoring

## Understanding peptide identification from MS2 spectra

Under low-energy fragmentation conditions, peptide fragment patterns are reproducible and, in general, predictable, which allows for an 
amino acid sequence identification according to a fragmentation expectation. Algorithms for peptide identification perform in principle 
three basic tasks:

**(i)** a raw data preprocessing step is applied to the MS/MS spectra to obtain clean peak information. The same signal filtering 
and background subtraction methods are used as discussed in the section of low-level processing. Peak detection, however, may be performed 
differently. Preprocessing of MS/MS spectra focuses on the extraction of the precise m/z of the peak rather than the accurate peak areas. 
The conversion of a peak profile into the corresponding m/z and intensity values reduces the complexity, its representation is termed centroiding. 
To extract the masses for identification in a simple and fast way, peak fitting approaches are used. These approaches take either the most intense 
point of the peak profile, fit a Lorentzian distribution to the profile, or use a quadratic fit. 

**(ii)** Spectrum information and possible amino acid sequences assignments are evaluated. 

**(iii)** The quality of the match between spectrum and possible sequences is scored.

Even though the three steps roughly describe the basic principle of algorithms used for peptide sequence identification, most implementations 
show differences in individual steps which can lead to major changes in the outcome. Therefore, it has been proven useful to utilize more than 
one algorithm for a robust and thorough identification. Due to their major difference in identification strategies and prerequisites, 
identification algorithms are normally classified into three categories: (i) *de novo* peptide sequencing, (ii) peptide sequence-tag (PST) 
searching, and (iii) uninterpreted sequence searching. However, in this notebook we focus on the explanation of (iii) uninterpreted sequence 
searching, the most frequently used methods.
## Matching and scoring of Tandem MS peptide identification

![](https://raw.githubusercontent.com/CSBiology/BIO-BTE-06-L-7/main/docs/img/ComputationalProteinIdentification.png)

**Figure 5: Process of computational identification of peptides from their fragment spectra**

Previously we learned, that peptides fragment to create patterns characteristic of a specific amino acid sequence. These patterns are reproducible and, in general, 
predictable taking the applied fragmentation method into account. This can be used for computational identification of peptides from their fragment spectra. 
This process can be subdivided into 5 main steps: spectrum preprocessing, selection of possible sequences, generating theoretical spectra, matching and scoring 
(Figure 5). The first step is a preprocessing of the experimental spectra and is done to reduce noise. Secondly, all possible amino acid 
sequences are selected which match the particular precursor peptide mass. The possible peptides can but do not need to be restricted to a particular organism. 
A theoretical spectrum is predicted for each of these amino acid sequences. Matching and scoring is performed by comparing experimental spectra to their predicted 
corresponding theoretical spectra. The score function measures the closeness of fit between the experimental acquired and theoretical spectrum. There are many 
different score functions, which can be considered as the main reason of different identifications considering different identification algorithm. The most 
natural score function is the cross correlation score (xcorr) used by the commercially available search tool SEQUEST. One of the reasons the xcorr is so 
sensitive is because it involves a correction factor that assesses the background correlation for each acquired spectrum and the theoretically predicted 
spectrum from sequences within a database. To compute this correction factor, a measure of similarity is calculated at different offsets between a preprocessed 
mass spectrum and a theoretical spectrum. The final xcorr is then the correlation at zero offset minus the mean correlation from all the individual offsets.

Thus, the correlation function measures the coherence of two signals by, in effect, translating one signal across the other. This can be mathematically 
achieved using the formula for cross-correlation in the form for discrete input signals:

***Equation 5***

![](https://latex.codecogs.com/png.latex?\large&space;R_{\tau}&space;=&space;\sum_{i=0}^{n-1}x_{i}\cdot%20y_{i&plus;\tau})

where x(i) is a peak of the reconstructed spectrum at position i and y(i) is a peak of the experimental spectrum. The displacement value 𝜏 
is the amount by which the signal is offset during the translation and is varied over a range of values. If two signals are the same, the correlation 
function should have its maxima at 𝜏=0, where there is no offset. The average of the offset-correlation is seen as the average background correlation 
and needs to be subtracted from the correlation at 𝜏=0. It follows: 

***Equation 6***

![](https://latex.codecogs.com/png.latex?xcorr&space;=&space;R_{0}&space;-&space;\frac{(\sum&space;\begin{matrix}&space;\tau=&plus;offeset\\&space;\tau=-offeset\end{matrix}R_{\tau})}{2*offset&plus;1})

In practice many theoretical spectra have to be matched again a single experimental spectrum. Therefore, the calculation can be speed up by reformulating Equation 5 and Equation 6 and introduce a preprocessing step, which is independent of the predicted spectra.

***Equation 7***

![](https://latex.codecogs.com/png.latex?xcorr&space;=&space;x_{0}\cdot&space;y_{0}&space;-&space;\frac{(\sum&space;\begin{matrix}&space;\tau=&plus;offeset\\&space;\tau=-offeset\end{matrix}x_{0}\cdot&space;y_{\tau})}{2*offset&plus;1})

For the preprocessed experimental spectrum y' it follows:

***Equation 8***

![](https://latex.codecogs.com/png.latex?xcorr&space;=&space;x_{0}\cdot&space;y`)

where:

***Equation 9***

![](https://latex.codecogs.com/png.latex?y'&space;=&space;y_{0}&space;-&space;\frac{(\sum&space;\begin{matrix}&space;\tau=&plus;offeset\\&space;\tau=-offeset\end{matrix}x_{0}\cdot&space;y_{\tau})}{2*offset&plus;1})

Matching a measured spectrum against chlamy database


In [1]:
#r "nuget: BioFSharp, 2.0.0-beta4"
#r "nuget: BioFSharp.IO, 2.0.0-beta4"
#r "nuget: Plotly.NET, 4.2.0"
#r "nuget: BioFSharp.Mz, 0.1.5-beta"
#r "nuget: MzIO, 0.1.7"
#r "nuget: MzIO.Processing, 0.1.2"
#r "nuget: MzLite, 1.1.0"
#r "nuget: Plotly.NET.Interactive, 4.2.0"
#r "nuget: MzIO.SQL, 0.1.9"


open Plotly.NET
open BioFSharp
open BioFSharp.Mz
open System.IO
open MzIO
open MzIO.Processing
open MzLite
open MzIO.MzSQL
open System.Data
open BioFSharp.Mz.TheoreticalSpectra
open BioFSharp.Mz.ChargeState



Installed Packages BioFSharp, 2.0.0-beta4 BioFSharp.IO, 2.0.0-beta4 BioFSharp.Mz, 0.1.5-beta MzIO, 0.1.7 MzIO.Processing, 0.1.2 MzIO.SQL, 0.1.9 MzLite, 1.1.0 Plotly.NET, 4.2.0 Plotly.NET.Interactive, 4.2.0

Loading extensions from `/home/paulinehans/.nuget/packages/plotly.net.interactive/4.2.0/lib/netstandard2.1/Plotly.NET.Interactive.dll`

### Step 1: Data acquisition and preprocessing

We start with reading one .mzlite run, filter MS2 scans, and collect per-scan information needed for peptide identification: scan ID, retention time, and fragment peak intensities (m/z & Intensity). 

In [2]:
//record type
type ms =  
    {
        mzIntensity: float array * float array
        retentiontime: float
        precursorInfo: list<int*float>
        id: string
    }

In [ ]:
// Code-Block 1

//read in mzmlite
let directory = __SOURCE_DIRECTORY__
let path = Path.Combine[|directory;"downloads/WC_Gr3_UV64 2.mzlite"|]

//set up connection between file and SQL database
let runID = "sample=0"
let mzReader = new MzIO.MzSQL.MzSQL(path)       // spectrum metadata reader
let peaksReader = new MzIO.MzSQL.MzSQL(path)    // peak data reader (m/z + intensity)
let cn = mzReader.Open()
let cn2 = peaksReader.Open()
let transaction = mzReader.BeginTransaction()



// Read all spectra in this run.
let getMs2 = mzReader.ReadMassSpectra runID

// Keep only MS2 spectra and extract the fields needed for identification.
let extractData =
    getMs2
    |> Seq.choose (fun x ->
        match MzIO.Processing.MassSpectrum.getMsLevel x with
        | 2 ->
            Some {
                id = x.ID
                precursorInfo = []
                retentiontime = MzIO.Processing.MassSpectrum.getScanTime x
                mzIntensity = 
                    PeakArray.mzIntensityArrayOf (
                        peaksReader.ReadSpectrumPeaks x.ID
                    )
            }
        | _ -> None
    )

    |> Seq.toArray

let allInfo = 
    extractData 
    |> Array.map (fun x -> x.id, x.precursorInfo, x.retentiontime, x.mzIntensity)
allInfo 

index value 0 (sample=1 period=1 cycle=16 experiment=2, [], 0.10065, (System.Double[], System.Double[])) Item1 sample=1 period=1 cycle=16 experiment=2 Item2 [ ] HeadOrDefault <null> TailOrNull <null> Head System.InvalidOperationException: The input list was empty.\n at Microsoft.FSharp.Collections.FSharpList`1.get_Head() in D:\a\_work\1\s\src\FSharp.Core\prim-types.fs:line 4319\n at lambda_method166(Closure, FSharpList`1)\n at Microsoft.DotNet.Interactive.Formatting.MemberAccessor`1.GetValueOrE... TargetSite T get_Head() Name get_Head DeclaringType Microsoft.FSharp.Collections.FSharpList<T> ReflectedType Microsoft.FSharp.Collections.FSharpList<T> MemberType Method MetadataToken 100663785 Module FSharp.Core.dll MDStreamVersion 131072 FullyQualifiedName /home/paulinehans/.nuget/packages/microsoft.dotnet-interactive/1.0.632301/tools/net9.0/any/FSharp.Core.dll ModuleVersionId 1023697b-e00e-d2ad-b568-6f799f8eeeea MetadataToken 1 ScopeName FSharp.Core.dll Name FSharp.Core.dll Assembly FSharp.Core, Version=9.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a ModuleHandle System.ModuleHandle CustomAttributes [ ] IsSecurityCritical True IsSecuritySafeCritical False IsSecurityTransparent False MethodHandle System.RuntimeMethodHandle Value 125016173735080 Attributes Public, HideBySig, SpecialName CallingConvention Standard, HasThis ReturnType T ReturnTypeCustomAttributes T ParameterType T Name <null> HasDefaultValue False DefaultValue RawDefaultValue MetadataToken 134217728 Attributes None Member T get_Head() Position -1 IsIn False IsLcid False IsOptional False IsOut False IsRetval False CustomAttributes [ ] ReturnParameter T ParameterType T Name <null> HasDefaultValue False DefaultValue RawDefaultValue MetadataToken 134217728 Attributes None Member T get_Head() Position -1 IsIn False IsLcid False IsOptional False IsOut False IsRetval False CustomAttributes [ ] IsCollectible False IsGenericMethod False IsGenericMethodDefinition False ContainsGenericParameters True MethodImplementationFlags IL IsAbstract False IsConstructor False IsFinal False IsHideBySig True IsSpecialName True IsStatic False IsVirtual False IsAssembly False IsFamily False IsFamilyAndAssembly False IsFamilyOrAssembly False IsPrivate False IsPublic True IsConstructedGenericMethod False CustomAttributes (empty) Message The input list was empty. Data (empty) InnerException <null> HelpLink <null> Source FSharp.Core HResult -2146233079 StackTrace at Microsoft.FSharp.Collections.FSharpList`1.get_Head() in D:\a\_work\1\s\src\FSharp.Core\prim-types.fs:line 4319
 at lambda_method166(Closure, FSharpList`1)
 at Microsoft.DotNet.Interactive.Formatting.MemberAccessor`1.GetValueOrException(T instance) in D:\a\_work\1\s\src\Microsoft.DotNet.Interactive.Formatting\MemberAccessor{T}.cs:line 58 Tail System.InvalidOperationException: The input list was empty.\n at Microsoft.FSharp.Collections.FSharpList`1.get_Tail() in D:\a\_work\1\s\src\FSharp.Core\prim-types.fs:line 4324\n at lambda_method167(Closure, FSharpList`1)\n at Microsoft.DotNet.Interactive.Formatting.MemberAccessor`1.GetValueOrE... TargetSite Microsoft.FSharp.Collections.FSharpList`1[T] get_Tail() Name get_Tail DeclaringType Microsoft.FSharp.Collections.FSharpList<T> ReflectedType Microsoft.FSharp.Collections.FSharpList<T> MemberType Method MetadataToken 100663786 Module FSharp.Core.dll MDStreamVersion 131072 FullyQualifiedName /home/paulinehans/.nuget/packages/microsoft.dotnet-interactive/1.0.632301/tools/net9.0/any/FSharp.Core.dll ModuleVersionId 1023697b-e00e-d2ad-b568-6f799f8eeeea MetadataToken 1 ScopeName FSharp.Core.dll Name FSharp.Core.dll Assembly FSharp.Core, Version=9.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a ModuleHandle System.ModuleHandle CustomAttributes [ ] IsSecurityCritical True IsSecuritySafeCritical False IsSecurityTransparent False MethodHandle System.RuntimeMethodHandle Value 125016173735104 Attributes Public, HideBySig, SpecialName CallingConvention Standard, HasThis ReturnType Microsoft.

Here, the spectrum is already centroidized as shown in *NB03c\_Centroidisation.ipynb* using the function 
`msPeakPicking`. So we pick a spectrum and just visualize mass and intensity:


In [4]:
// Code-Block 2

//first we only spectrum as example
let exampleSpectrum = 
    allInfo 
    |> Array.find (fun (ID, precursorInfo, rt, (mz, intensity)) -> 
        ID.Contains("sample=1 period=1 cycle=3873 experiment=13"))

let exampleData1 = 
    exampleSpectrum 
    |> fun (x,y,z,a) -> fst a 

let exampleData2 = 
    exampleSpectrum 
    |> fun (x,y,z,a) -> snd a 


//creating chart, which visualizes m/z & intensity
let ms2Chart =  
    let chart = 
        Chart.Column(values = exampleData2, Keys = exampleData1,  Width = 5.)
        |> Chart.withYAxisStyle("Intensity")
        |> Chart.withXAxisStyle("m/z")
        |> Chart.withSize(1200, 800)
    chart
ms2Chart


<!-- Plotly chart will be drawn inside this DIV -->

Looks nice, but there are some overlapping columns, which we now try to get rid of, by using the function: peaksToNearestUnitDaltonBinVector. You'll see (hopefully) that the columns are then more sperated. Think about why this could be necessary. 

In [ ]:
// Code-Block 3

// Define m/z range for binning (in Dalton).
let lowerScanLimit = 150.
let upperScanLimit = 1000.

//// Keep only the fields needed downstream: ID, RT, m/z array, intensity array.
let makingDataPretty = 
    allInfo 
    |> Array.map (fun (ID, _, rt, (mz, intensity)) -> ID,rt,  mz, intensity)

//normalization and preprocess each MS2 spectrum:
let preprocessedIntesities = 
    makingDataPretty 
    |> Array.map (fun (ID,rt, mz,int) -> 
        let normalization =
            Mz.PeakArray.zip mz int
            |> (fun pa -> Mz.PeakArray.peaksToNearestUnitDaltonBinVector pa lowerScanLimit upperScanLimit)
            |> (fun pa -> Mz.SequestLike.windowNormalizeIntensities pa 10)
        ID,rt,normalization
    )
preprocessedIntesities


//example Spectrum Chart
let exampleData = 
    preprocessedIntesities
    |> Array.find (fun (ID, rt, vector) -> 
        ID.Contains("sample=1 period=1 cycle=3873 experiment=13"))
preprocessedIntesities
// Unpack selected spectrum: id, retention time, normalized intensity vector.
let (id, rt, vec) = exampleData

// Convert vector index back to m/z value using lowerScanLimit as offset.
let keysValues =
    vec
    |> Seq.mapi (fun i intensity ->
        let mz = lowerScanLimit + float i   
        (mz, intensity)
    )
    |>Seq.toArray
    
let exampleChart = 
        Chart.Column (keysValues = keysValues , Width = 1. )
        |> Chart.withSize(1400,800)
    
exampleChart

<!-- Plotly chart will be drawn inside this DIV -->

Phew ... nice the binning worked!

Now, we will preprocess the experimental spectrum from our .mzlite file. This is on the one hand to reduce noise, but also to make 
the measured spectrum more like the once we are able to simulate. 


### Step 2: Preselecting the peptides of interest
From our previos notebook *NB02b\_Digestion\_and\_mass\_calculation.ipynb*, we know how to 
calculate all peptide masses that we can expect to be present in *Chlamydomonas reinhardtii*.




In [6]:
// Code-Block 4
let path2 = Path.Combine[|directory;"downloads/Chlamy_JGI5_5_QProt.fasta"|]

let peptideAndMasses = 
    path2
    |> IO.FastA.fromFile BioArray.ofAminoAcidString
    |> Seq.toArray
    |> Array.mapi (fun i fastAItem ->
        Digestion.BioArray.digest Digestion.Table.Trypsin i fastAItem.Sequence
        |> Digestion.BioArray.concernMissCleavages 0 0
    )
    |> Array.concat
    |> Array.map (fun peptide ->
        // calculate mass for each peptide
        peptide.PepSequence, BioSeq.toMonoisotopicMassWith (BioItem.monoisoMass ModificationInfo.Table.H2O) peptide.PepSequence
    )

peptideAndMasses |> Array.head


([Leu; Ala; Ala; ... ], 1408.6860982223602) Item1 [ Leu, Ala, Ala, Ala, Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Leu TailOrNull [ Ala, Ala, Ala, Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Ala TailOrNull [ Ala, Ala, Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Ala TailOrNull [ Ala, Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Ala TailOrNull [ Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Leu TailOrNull [ Glu, His, His, His, His, His, His ] Head Leu Tail [ Glu, His, His, His, His, His, His ] (values) index value 0 Leu 1 Glu 2 His 3 His 4 His 5 His 6 His 7 His Head Ala Tail [ Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Leu TailOrNull [ Glu, His, His, His, His, His, His ] Head Leu Tail [ Glu, His, His, His, His, His, His ] (values) index value 0 Leu 1 Glu 2 His 3 His 4 His 5 His 6 His 7 His (values) index value 0 Ala 1 Leu 2 Glu 3 His 4 His 5 His 6 His 7 His 8 His Head Ala Tail [ Ala, Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Ala TailOrNull [ Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Leu TailOrNull [ Glu, His, His, His, His, His, His ] Head Leu Tail [ Glu, His, His, His, His, His, His ] (values) index value 0 Leu 1 Glu 2 His 3 His 4 His 5 His 6 His 7 His Head Ala Tail [ Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Leu TailOrNull [ Glu, His, His, His, His, His, His ] Head Leu Tail [ Glu, His, His, His, His, His, His ] (values) index value 0 Leu 1 Glu 2 His 3 His 4 His 5 His 6 His 7 His (values) index value 0 Ala 1 Leu 2 Glu 3 His 4 His 5 His 6 His 7 His 8 His (values) index value 0 Ala 1 Ala 2 Leu 3 Glu 4 His 5 His 6 His 7 His 8 His 9 His Head Ala Tail [ Ala, Ala, Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Ala TailOrNull [ Ala, Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Ala TailOrNull [ Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Leu TailOrNull [ Glu, His, His, His, His, His, His ] Head Leu Tail [ Glu, His, His, His, His, His, His ] (values) index value 0 Leu 1 Glu 2 His 3 His 4 His 5 His 6 His 7 His Head Ala Tail [ Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Leu TailOrNull [ Glu, His, His, His, His, His, His ] Head Leu Tail [ Glu, His, His, His, His, His, His ] (values) index value 0 Leu 1 Glu 2 His 3 His 4 His 5 His 6 His 7 His (values) index value 0 Ala 1 Leu 2 Glu 3 His 4 His 5 His 6 His 7 His 8 His Head Ala Tail [ Ala, Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Ala TailOrNull [ Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Leu TailOrNull [ Glu, His, His, His, His, His, His ] Head Leu Tail [ Glu, His, His, His, His, His, His ] (values) index value 0 Leu 1 Glu 2 His 3 His 4 His 5 His 6 His 7 His Head Ala Tail [ Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Leu TailOrNull [ Glu, His, His, His, His, His, His ] Head Leu Tail [ Glu, His, His, His, His, His, His ] (values) index value 0 Leu 1 Glu 2 His 3 His 4 His 5 His 6 His 7 His (values) index value 0 Ala 1 Leu 2 Glu 3 His 4 His 5 His 6 His 7 His 8 His (values) index value 0 Ala 1 Ala 2 Leu 3 Glu 4 His 5 His 6 His 7 His 8 His 9 His (values) index value 0 Ala 1 Ala 2 Ala 3 Leu 4 Glu 5 His 6 His 7 His 8 His 9 His 10 His Head Leu Tail [ Ala, Ala, Ala, Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Ala TailOrNull [ Ala, Ala, Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Ala TailOrNull [ Ala, Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Ala TailOrNull [ Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Leu TailOrNull [ Glu, His, His, His, His, His, His ] Head Leu Tail [ Glu, His, His, His, His, His, His ] (values) index value 0 Leu 1 Glu 2 His 3 His 4 His 5 His 6 His 7 His Head Ala Tail [ Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Leu TailOrNull [ Glu, His, His, His, His, His, His ] Head Leu Tail [ Glu, His, His, His, His, His, His ] (values) index value 0 Leu 1 Glu 2 His 3 His 4 His 5 His 6 His 7 His (values) index value 0 Ala 1 Leu 2 Glu 3 His 4 His 5 His 6 His 7 Hi

In the previous notebook *NB04a\_Fragmentation\_for\_peptide\_identification.ipynb*, 
we used functions that generate the theoretical series of b- and y-ions from the given peptide. Combined with the function 
`Mz.SequestLike.predictOf` that generates theoretical spectra that fit the Sequest scoring algorithm.


In [7]:
// Code-Block 6

let predictFromSequence peptide charge =
    [
        peptide
        |> Mz.Fragmentation.Series. yOfBioList BioItem.initMonoisoMassWithMemP
        peptide
        |> Mz.Fragmentation.Series.bOfBioList BioItem.initMonoisoMassWithMemP
    ]
    |> List.concat
    |> Mz.SequestLike.predictOf (lowerScanLimit,upperScanLimit) charge 
 

### Step 3+4: Matching and Scoring

In the matching step, we compare theoretical spectra of peptides within our precursor peptide mass range with our measured spectra. 
We get a score which tells us how well the theoretical spectrum fits the given experimental spectrum.


In [8]:
let numbers : ChargeState.ChargeDetermParams =
    {
        ExpectedMinimalCharge = 2
        ExpectedMaximumCharge = 5
        Width = 1.1
        MinIntensity = 0.15
        DeltaMinIntensity = 0.3
        NrOfRndSpectra = 1000
    }

let rnd = new System.Random()

Following you have a helper function (Don't be alarmed by the length lol) you dont need to understand in detail was the function is doing but here is a short explanation:

The getPrecursorCharge rnd helper function determines the possible charge states of the corresponding precursor ion for each MS2 spectrum. To do this, all mass spectra of a run are first read in and divided into MS1 and MS2 spectra, which are then sorted according to their scan time. Each MS2 spectrum is then assigned to the preceding MS1 spectrum, as the fragmented precursor was measured in the corresponding MS1 spectrum. In the next step, the isotope pattern of the precursor is analyzed in the corresponding MS1 spectrum to determine possible charge states. If no charge can be clearly identified, typical charge states, such as 2+ and 3+, are assumed as a fallback. Finally, all possible combinations of charge and calculated mass are collected and returned as the result.

The <b>important question</b> you should aks yourself here is, <b>why is the charge so important?</b>

In [ ]:
let getPrecursorCharge rnd =
    /// Returns a Sequence containing all MassSpectra of a single MS-run
    let massSpectra = mzReader.ReadMassSpectra(runID) 
    /// Returns a Array that contains all MS1s sorted by their scanTime
    let ms1SortedByScanTime =
        massSpectra
        |> Seq.filter (fun ms -> MassSpectrum.getMsLevel ms = 1)
        |> Seq.map (fun ms -> MassSpectrum.getScanTime ms, ms)
        |> Seq.sortBy fst
        |> Array.ofSeq
    let ms2SortedByScanTime =
        massSpectra
        |> Seq.filter (fun ms -> MassSpectrum.getMsLevel ms = 2)
        |> Seq.map (fun ms -> MassSpectrum.getScanTime ms, ms)
        |> Seq.sortBy fst
        |> Array.ofSeq
    let ms2AssignedToMS1 =
        ms2SortedByScanTime
        |> Array.choose (fun (ms2ScanTime,ms2) ->
            let ms1' =
                ms1SortedByScanTime
                |> Array.tryFindBack (fun (ms1ScanTime,_) ->
                    ms1ScanTime <= ms2ScanTime)
            match ms1' with 
            | Some (_,ms1) -> 
                Some (ms1, ms2)
            | None -> None 
            )
        |> Array.groupBy fst
        |> Array.map (fun (ms1,ms1And2) -> ms1,ms1And2 |> Array.map snd)
    
    let ms2PossibleChargestates =
        ms2AssignedToMS1
        |> Array.mapi (fun i (ms1,ms2s) ->
            //printfn "%i" i
            let mzdata,intensityData = MzIO.Processing.PeakArray.mzIntensityArrayOf (mzReader.ReadSpectrumPeaks(ms1.ID))
            ms2s
            |> Array.filter (fun ms2 -> mzReader.ReadSpectrumPeaks(ms2.ID).Peaks |> Seq.isEmpty = false)
            |> Array.map (fun ms2 ->
                                let assignedCharges = ChargeState.putativePrecursorChargeStatesBy numbers mzdata intensityData ms1.ID ms2.ID (MassSpectrum.getPrecursorMZ ms2)
                                match assignedCharges with
                                | [] ->
                                    [
                                        for i = 2 to 3 do
                                            let precursorMz = (MassSpectrum.getPrecursorMZ ms2)
                                            let mass = Mass.ofMZ precursorMz (float i)
                                            let score = getScore 10 1 100.
                                            yield createAssignedCharge ms1.ID ms2.ID precursorMz i mass 100. score [0.] 0 0. (Set[]) (Some 1.)
                                    ]
                                    |> Array.ofList
                                | _ -> assignedCharges |> Array.ofList
                            )
            )
        |> Array.filter (fun x ->  Array.isEmpty x |> not)
        |> Array.concat
        |> Array.filter (fun (assignedCharges)  -> assignedCharges.Length = 0 |> not)
    ms2PossibleChargestates


let exe = getPrecursorCharge rnd
exe

let collectChargeMz =
    exe
    |> Array.collect (fun (assignedCharges) ->
                    assignedCharges
                    |> Array.map (fun assCh -> assCh.ProductSpecID, (assCh.PrecCharge, assCh.PutMass))
                )
    |> Array.groupBy (fun (ms2Id,assCH) -> ms2Id)
    |> Array.map (fun (x,y) -> x, y |> Array.map snd)
    |> Map.ofArray
collectChargeMz

After we collect everything important we combine and format it, so that it will be easier to work with it

In [ ]:
// Combine normalized MS2 spectra with precursor/charge information from collectChargeMz.
// For each spectrum: keep id, retention time, normalized MS2 vector, and matched precursor info.
let wholeInfo = 
    preprocessedIntesities
    |> Array.map (fun (id,rt,ms2) -> 
        id,rt, ms2,(collectChargeMz.TryFind(id)).Value
    )
wholeInfo

index value 0 (sample=1 period=1 cycle=3996 experiment=2, 64.678433333333, vector [0;0;0;0;0;1;0.1955991554;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0.1342096623;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0.09699326764;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0.1223311511;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0.0721447347;0;0;0... Item1 sample=1 period=1 cycle=3996 experiment=2 Item2 64.678433333333 Item3 [ 0, 0, 0, 0, 0, 1, 0.19559915536881886, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0 ... (more) ] InternalValues [ 0, 0, 0, 0, 0, 1, 0.19559915536881886, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0 ... (830 more) ] Values [ 0, 0, 0, 0, 0, 1, 0.19559915536881886, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0 ... (830 more) ] OpsData Some(FSharp.Stats.Instances+FloatNumerics@116) Value FSharp.Stats.Instances+FloatNumerics@116 Length 850 NumRows 850 ElementOps FSharp.Stats.Instances+FloatNumerics@116 DebugDisplay vector [0;0;0;0;0;1;0.1955991554;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0.1342096623;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0.09699326764;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0.1223311511;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0.0721447347;0;0;0;0;0.3586888792;0;0;0;0;0;0;0;0;0;0; ...] Capacity 280 MaxCapacity 2147483647 Length 280 StructuredDisplayAsArray [ 0, 0, 0, 0, 0, 1, 0.19559915536881886, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0 ... (830 more) ] Details [ 0, 0, 0, 0, 0, 1, 0.19559915536881886, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0 ... (830 more) ] Transpose [ 0, 0, 0, 0, 0, 1, 0.19559915536881886, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0 ... (more) ] InternalValues [ 0, 0, 0, 0, 0, 1, 0.19559915536881886, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0 ... (830 more) ] Values [ 0, 0, 0, 0, 0, 1, 0.19559915536881886, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0 ... (830 more) ] OpsData Some(FSharp.Stats.Instances+FloatNumerics@116) Value FSharp.Stats.Instances+FloatNumerics@116 Length 850 NumCols 850 ElementOps FSharp.Stats.Instances+FloatNumerics@116 DebugDisplay rowvec [0;0;0;0;0;1;0.1955991554;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0.1342096623;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0.09699326764;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0.1223311511;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0.0721447347;0;0;0;0;0.3586888792;0;0;0;0;0;0;0;0;0;0; ...] Capacity 280 MaxCapacity 2147483647 Length 280 StructuredDisplayAsArray [ 0, 0, 0, 0, 0, 1, 0.19559915536881886, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0 ... (830 more) ] Details [ 0, 0, 0, 0, 0, 1, 0.19559915536881886, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0 ... (830 more) ] Transpose [ 0, 0, 0, 0, 0, 1, 0.19559915536881886, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0 ... (more) ] InternalValues [ 0, 0, 0, 0, 0, 1, 0.19559915536881886, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0 ... (830 more) ] Values [ 0, 0, 0, 0, 0, 1, 0.19559915536881886, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0 ... (830 more) ] OpsData Some(FSharp.Stats.Instances+FloatNumerics@116) Value FSharp.Stats.Instances+FloatNumerics@116 Length 850 NumRows 850 ElementOps FSharp.Stats.Instances+FloatNumerics@116 DebugDisplay vector [0;0;0;0;0;1;0.1955991554;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0.1342096623;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0.09699326764;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0.1223311511;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0.0721447347;0;0;0;0;0.3586888792;0;0;0;0;0;0;0;0;0;0; ...] Capacity 280 MaxCapacity 2147483647 Length 280 StructuredDisplayAsArray [ 0, 0, 0, 0, 0, 1, 0.19559915536881886, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0 ... (830 more) ] Details [ 0, 0, 0, 0, 0, 1, 0.19559915536881886, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0 ... (830 more) ] Transpose [ 0, 0, 0, 0, 0, 1, 0.19559915536881886, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0 ... (more) ] InternalValues [ 0, 0, 0, 0, 0, 1, 0.19559915536881886, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0 ... (830 more) ] Values [ 0, 0, 0, 0, 0, 1, 0.19559915536881886, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0 ... (830 more) ] OpsData Some(FSharp.Stats.Instances+FloatNumerics@116) Length 850 NumCols 850 ElementOps FSharp.Stats.Instances+FloatNumeri

What happens now is the actual core process, the <b>peptide identification</b> for each MS2 spectrum. For each spectrum, possible precursor charges are tested, theoretical peptides are generated, compared with the experimental spectrum, and finally the best-rated match per spectrum is retained.

In [ ]:
// For each MS2 entry, keep only spectra where at least one peptide candidate can be scored.
let spectras = 
    wholeInfo 
    |> Array.choose (fun (ID,rt, vec,possibleChargesAndMasses) -> 
        // For this spectrum, evaluate each possible precursor charge/mass pair.
        possibleChargesAndMasses
        |> Array.choose (fun (charge,mass) ->
            // Score peptide candidates for one charge state and keep the top-scoring hit.
            let bestScoredPeptideOneCharge = 
                peptideAndMasses
                // Coarse precursor mass filter to reduce the search space.
                |> Array.filter (fun (_,massTheo) -> abs(massTheo-mass) <= 0.5 )
                |> Array.map (fun (sequence,mass) ->
                    // Generate theoretical fragments and compare them to the measured spectrum.
                    let theoSpec = predictFromSequence sequence charge
                    sequence,charge,mass,BioFSharp.Mz.SequestLike.scoreSingle theoSpec vec
                )
                // Rank candidates by score and keep only the best one for this charge.
                |> Array.sortByDescending (fun (_,_,_,score) -> score)
                |> Array.tryHead
            // Convert successful hit to output tuple; skip charge states with no candidates.
            match bestScoredPeptideOneCharge with
            | Some (aaList,charge,mass,score) -> 
                Some {|SpectrumID = ID; RetentionTime = rt; PeptideSequence = aaList |> BioList.toString; Charge = charge; Mass = mass; MatchingScore = score|}
            | None -> None

        )
        // From all tested charge states, keep the single best hit per spectrum.
        |> Array.sortByDescending (fun result -> result.MatchingScore)
        |> Array.tryHead
    )
spectras



index value 0 { Charge = 2\n Mass = 1295.756135\n MatchingScore = 0.7645020568\n PeptideSequence = "LQNIVGVPTSIR"\n RetentionTime = 63.80898333\n SpectrumID = "sample=1 period=1 cycle=3887 experiment=5" } Charge 2 Mass 1295.75613484367 MatchingScore 0.7645020568323313 PeptideSequence LQNIVGVPTSIR RetentionTime 63.808983333333 SpectrumID sample=1 period=1 cycle=3887 experiment=5 1 { Charge = 2\n Mass = 1458.725563\n MatchingScore = 0.0\n PeptideSequence = "LIFQYASFNNSR"\n RetentionTime = 63.78498333\n SpectrumID = "sample=1 period=1 cycle=3886 experiment=12" } Charge 2 Mass 1458.7255629902602 MatchingScore 0 PeptideSequence LIFQYASFNNSR RetentionTime 63.784983333333 SpectrumID sample=1 period=1 cycle=3886 experiment=12 2 { Charge = 2\n Mass = 1362.662688\n MatchingScore = 1.169364997\n PeptideSequence = "GILASDESNATTGK"\n RetentionTime = 63.78165\n SpectrumID = "sample=1 period=1 cycle=3886 experiment=10" } Charge 2 Mass 1362.6626879473802 MatchingScore 1.169364996777007 PeptideSequence GILASDESNATTGK RetentionTime 63.78165 SpectrumID sample=1 period=1 cycle=3886 experiment=10 3 { Charge = 2\n Mass = 920.4967323\n MatchingScore = 0.3234160189\n PeptideSequence = "FIESQVAK"\n RetentionTime = 63.76831667\n SpectrumID = "sample=1 period=1 cycle=3886 experiment=2" } Charge 2 Mass 920.49673227576 MatchingScore 0.3234160188668125 PeptideSequence FIESQVAK RetentionTime 63.768316666667 SpectrumID sample=1 period=1 cycle=3886 experiment=2 4 { Charge = 2\n Mass = 1502.856912\n MatchingScore = 0.0\n PeptideSequence = "SVVSIPHGPSIIAAR"\n RetentionTime = 63.75015\n SpectrumID = "sample=1 period=1 cycle=3885 experiment=16" } Charge 2 Mass 1502.85691151298 MatchingScore 0 PeptideSequence SVVSIPHGPSIIAAR RetentionTime 63.75015 SpectrumID sample=1 period=1 cycle=3885 experiment=16 5 { Charge = 2\n Mass = 1203.665195\n MatchingScore = 1.210770849\n PeptideSequence = "TPLANLVYWK"\n RetentionTime = 63.74181667\n SpectrumID = "sample=1 period=1 cycle=3885 experiment=6" } Charge 2 Mass 1203.66519458263 MatchingScore 1.2107708485725233 PeptideSequence TPLANLVYWK RetentionTime 63.741816666667 SpectrumID sample=1 period=1 cycle=3885 experiment=6 6 { Charge = 2\n Mass = 1025.648482\n MatchingScore = 2.91265903\n PeptideSequence = "AVSLVLPSLK"\n RetentionTime = 63.71033333\n SpectrumID = "sample=1 period=1 cycle=3884 experiment=3" } Charge 2 Mass 1025.64848188989 MatchingScore 2.912659030109447 PeptideSequence AVSLVLPSLK RetentionTime 63.710333333333 SpectrumID sample=1 period=1 cycle=3884 experiment=3 7 { Charge = 2\n Mass = 1183.717624\n MatchingScore = 0.1615533752\n PeptideSequence = "LSELLGKPVTK"\n RetentionTime = 63.65216667\n SpectrumID = "sample=1 period=1 cycle=3882 experiment=5" } Charge 2 Mass 1183.7176240771903 MatchingScore 0.1615533751587192 PeptideSequence LSELLGKPVTK RetentionTime 63.652166666667 SpectrumID sample=1 period=1 cycle=3882 experiment=5 8 { Charge = 3\n Mass = 1296.703765\n MatchingScore = 0.0\n PeptideSequence = "LVDELNAGTIPR"\n RetentionTime = 63.64966667\n SpectrumID = "sample=1 period=1 cycle=3882 experiment=2" } Charge 3 Mass 1296.7037649165197 MatchingScore 0 PeptideSequence LVDELNAGTIPR RetentionTime 63.649666666667 SpectrumID sample=1 period=1 cycle=3882 experiment=2 9 { Charge = 2\n Mass = 1295.756135\n MatchingScore = 0.0\n PeptideSequence = "LQNIVGVPTSIR"\n RetentionTime = 63.62931667\n SpectrumID = "sample=1 period=1 cycle=3881 experiment=13" } Charge 2 Mass 1295.75613484367 MatchingScore 0 PeptideSequence LQNIVGVPTSIR RetentionTime 63.629316666667 SpectrumID sample=1 period=1 cycle=3881 experiment=13 10 { Charge = 2\n Mass = 1253.531497\n MatchingScore = 1.369082226\n PeptideSequence = "MASMTGGQQMGR"\n RetentionTime = 63.62516667\n SpectrumID = "sample=1 period=1 cycle=3881 experiment=8" } Charge 2 Mass 1253.53149726641 MatchingScore 1.3690822262780644 PeptideSequence MASMTGGQQMGR RetentionTime 63.625166666667 SpectrumID sample=1 period=1 cycle=3881 experiment=8 11 { Charge = 2\n Mass = 1064.513839\n MatchingScore

So short recap: we compared all experimental spectra in the mzlite file with theoretical spectra from the FASTA file. Now we take the best matching scores. This will tell us, the most probable peptide sequence.
WONDERFUL its done (yay!) but notice however, that in a real world we would need to 
relate our score to the complete data set to get an idea of the overall quality and which numerical value we could trust. 

The last thing we do is to save our data in a .tsv file 

In [ ]:
spectras
|> FSharpAux.IO.SeqIO.Seq.CSV "\t" true false
|> fun csv -> System.IO.File.WriteAllLines("/home/paulinehans/Dokumente/scoring.tsv", csv)

## Questions

1. How does peptide identification generally work in this workflow, from raw data to identified peptide? You dont have do be super detailed
2. What is the difference between MS1 and MS2 spectra, and what role do they play in peptide identification?
3. Why is it necessary to consider the charge state of a precursor when determining the mass of a peptide?
4. What does the score calculated when comparing the theoretical and experimental spectra indicate?
5. What sources of error can lead to a spectrum being incorrectly identified or not identified at all?